# PC-FNO3D NATIONAL — phases 6 (agriculture) et 7 (éolien) : produits et critères, sans entraînement (Colab, A100)

**Exécution uniquement avec l'accord écrit d'Ines.** Aucun entraînement, aucun téléchargement, aucun
réglage : le carnet applique deux règles **PROPOSITION**, écrites et hachées avant tout calcul et vérifiées
par la première cellule :

| Règle | sha256 |
|---|---|
| `configs/national/regle_agri_national.json` (AGRI1, AGRI2, J0–J+2, FAO-56) | `739467e28a91…` |
| `configs/national/regle_eolien_national.json` (carte 50–150 m, zoom, EOL2) | `ff0b869c33a6…` |

Code figé au commit `f31bacc5d7e7` du dépôt public (à pousser avant l'exécution).

**Modèle** : le modèle national livré, désigné par `runs/decision_v4_national.json` (entraînement choisi :
`runs/choix_v4.json`), sinon la v2 ; on peut l'imposer avec `MODELE` (cellule 3). Sorties dans un dossier
neuf, `MyDrive/pcfno3d_national/phase6_7/<run>/` ; chaque script refuse de recalculer une sortie existante.

- **Phase 6** : AGRI1 / AGRI2 aux 12 aéroports (test 2022, chaque station retirée de l'entrée quand elle est
  jugée : post-traitement des paires du carnet de test v4, aucune nouvelle passe) ; fenêtres de traitement et
  d'aspersion J0–J+2 et besoin en eau FAO-56 sur la grille de 1 km, vent à 2 et 10 m au-dessus du sol
  (rejeu sur analyse ERA5 tant que la phase 4 n'a pas fourni le forçage GFS national).
- **Phase 7** : carte du potentiel de 50 à 150 m (vitesse, densité de puissance, Weibull, production d'une
  éolienne générique de 2 MW ; GeoTIFF, PNG, CSV) sur la validation 2021 et le test 2022 (instants jamais vus
  à l'entraînement ; hiver et été seulement, c'est dit), carte d'information sur les 1 000 instants, ERA5
  seule à côté ; zoom Grand Tunis à 100 m depuis l'atlas du modèle livré v4 ; EOL2 contre les radiosondages
  0–250 m (Tunis-Carthage, Tozeur).

ERA5 et la vérité générée sont la **vérité d'entraînement**, jamais des mesures ; les METAR et les
radiosondages sont les mesures. La session s'arrête à la fin.

## 1. Drive, code figé, règles vérifiées, données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit FIGE (les scripts des phases 6 et 7 y sont) ; regles verifiees
COMMIT = "f31bacc5d7e7b6fc9ed0476c273637db57d6e85d"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
# regles PROPOSITION ecrites et hachees AVANT tout calcul : une regle modifiee arrete le carnet
for f, h in {"regle_agri_national.json": "739467e28a911faded8249f725da2b5a790ebdca79afe54f9d6598e054d37b85", "regle_eolien_national.json": "ff0b869c33a638aad170fca77a1c73372e0c693d9c13520afdc74d279cfdbc6a"}.items():
    lu = hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest()
    assert lu == h, f"{f} : empreinte {lu} != {h} (regle modifiee depuis son hachage)"
print("code", COMMIT, "| regles verifiees :", ", ".join(f"{f} {h[:12]}" for f, h in {"regle_agri_national.json": "739467e28a911faded8249f725da2b5a790ebdca79afe54f9d6598e054d37b85", "regle_eolien_national.json": "ff0b869c33a638aad170fca77a1c73372e0c693d9c13520afdc74d279cfdbc6a"}.items()))

## 2. Cache v3 (3 parties du Drive)

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
lancer("controle", "python -u -m pcfno3d -c national_1km_colab check")

## 3. Modèle national retenu et configuration de production

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

# ----- modele national : celui que designe la decision v4 (runs/decision_v4_national.json), sinon la v2 -----
MODELE = None          # None = la decision ; ou un nom de run impose, ex. "national_1km_v4_Cp_s43" (ecrit dans les sorties)
RUNS = NAT / "runs"
DEC = RUNS / "decision_v4_national.json"
if MODELE is None:
    MODELE = json.loads(DEC.read_text())["modele_livre_national"] if DEC.exists() else "national_1km_v2"
    print("decision v4 nationale :", DEC if DEC.exists() else "absente -> modele livre v2")
if (RUNS / "choix_v4.json").exists():
    print("choix_v4.json (entrainement choisi sur la validation) :",
          json.loads((RUNS / "choix_v4.json").read_text())["entrainement_choisi"])
RUN = RUNS / MODELE
# la v2 jugee sur la verite v3 (memes poids, dossier neuf du carnet de test v4) porte les paires du test v3
if MODELE == "national_1km_v2" and (RUNS / "national_1km_v2_sur_verite_v3" / "best.pt").exists():
    RUN = RUNS / "national_1km_v2_sur_verite_v3"
CKPT = choisir_poids(RUN)
SORTIE = NAT / "phase6_7" / RUN.name           # dossier neuf, a part du run_dir ; rien n'est ecrase
SORTIE.mkdir(parents=True, exist_ok=True)
OBS = G / "obs"                                 # METAR des 12 aeroports, radiosondages IGRA
import yaml
c = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))
c["name"] = "national_phase67"
c["paths"]["run_dir"], c["paths"]["local_dir"] = str(RUN), "/content/run_local_phase67"
c["paths"]["stations_obs"] = str(OBS) if OBS.exists() else None     # entree de production : vrais METAR
c.setdefault("inference", {})["w_continuite"] = True
yaml.safe_dump(c, open(CODE / "configs" / "national_phase67.yaml", "w"), sort_keys=False)
print("modele :", MODELE, "| run :", RUN, "| poids :", CKPT.name, "| sorties :", SORTIE)
print("mesures :", sorted(p.name for p in OBS.iterdir()) if OBS.exists() else "AUCUN dossier obs")

## 4. Phase 6 : AGRI1 / AGRI2, fenêtres J0–J+2, FAO-56

In [ ]:
import shlex
from google.colab import runtime
import traceback
ARRETER = True          # liberer la machine a la fin de la phase 7 (cellule suivante)
try:
    # AGRI1 / AGRI2 : paires « station retiree » du test 2022 (ecrites UNE fois par le carnet de test v4) ;
    # si elles manquent pour ce modele, le meme script les ecrit une fois
    PAIRES = RUN / "validation_reelle" / "test" / "stations_paires.csv"
    if not PAIRES.exists():
        lancer("validation_reelle_phase6", f"python -u scripts/valider_grand_tunis_reel.py -c national_phase67 "
                                           f"--checkpoint {shlex.quote(str(CKPT))} --obs {OBS} --jeu test")
    D = SORTIE / "agriculture" / "agri1_agri2"
    if not (D / "verdict_agri_national.json").exists():
        lancer("agri1_agri2", f"python -u scripts/agri_national.py decisions --run {shlex.quote(str(RUN))} --sortie {D}")
    print((D / "verdict_agri_national.md").read_text())

    # fenetres J0-J+2 et FAO-56 sur la grille nationale (jours de base de la regle : validation 2021)
    GFS = next((f for f in (NAT / "gfs_tunisie_f24.npz", GT / "os7" / "gfs_tunisie_f24.npz") if f.exists()), None)
    print("cube GFS national :", GFS or "absent -> repli signale (T, HR : verite d'entrainement ; Rs = 0,72 Ra)")
    P = SORTIE / "agriculture" / "produit_j0_j2"
    if not (P / "produit_agricole.json").exists():
        lancer("produit_agricole", f"python -u scripts/agri_national.py produit -c national_phase67 "
                                   f"--checkpoint {shlex.quote(str(CKPT))} --sortie {P}"
                                   + (f" --gfs {GFS}" if GFS else ""))
    print((P / "produit_agricole.json").read_text()[:4000])
except Exception:
    traceback.print_exc()

## 5. Phase 7 : carte 50–150 m, zoom Grand Tunis 100 m, EOL2 — puis fin de la session

In [ ]:
try:
    # carte du potentiel 50-150 m (principale : validation + test ; information : tous les instants)
    A = SORTIE / "eolien" / "atlas_national"
    if not (A / "atlas_national.json").exists():
        TERR = NAT / "territoire_tunisie.json"     # polygone [[lat, lon], ...] facultatif, depose par Ines
        lancer("atlas_national", f"python -u scripts/atlas_eolien_national.py carte -c national_phase67 "
                                 f"--checkpoint {shlex.quote(str(CKPT))} --sortie {A}"
                                 + (f" --territoire {TERR}" if TERR.exists() else ""))
    print(pd.read_csv(A / "resume.csv").to_string(index=False))
    print(pd.read_csv(A / "zones.csv").to_string(index=False))

    # zoom Grand Tunis a 100 m : l'atlas du modele livre v4 (carnet OS6), aucune nouvelle passe
    Z = SORTIE / "eolien" / "zoom_grand_tunis_100m"
    if not (Z / "zoom.json").exists():
        lancer("zoom_grand_tunis", f"python -u scripts/atlas_eolien_national.py zoom "
                                   f"--grand-tunis {GT / 'runs' / 'atlas_eolien'} --national {A} "
                                   f"--decision-gt {GT / 'runs' / 'decision_v4.json'} --sortie {Z}")
    print((Z / "zoom.json").read_text()[:3000])

    # EOL2 : radiosondages 0-250 m (Tunis-Carthage, Tozeur s'ils sont dans obs) contre ERA5, test 2022
    E = SORTIE / "eolien" / "eol2"
    if not (E / "eol2.json").exists():
        lancer("eol2_national", f"python -u scripts/eol2_national.py -c national_phase67 "
                                f"--checkpoint {shlex.quote(str(CKPT))} --obs {OBS} --sortie {E}")
    print((E / "eol2.md").read_text())
except Exception:
    traceback.print_exc()
finally:
    if ARRETER:
        runtime.unassign()